# 🐄🔍 Cow Identifier — who is who

Takes your **already-trained** cow detector (`weights/best.pt`) and adds a second stage that
tells individual cows apart by their coat pattern. Nothing is retrained.

**Set up first:** a `looking_for/` folder with one sub-folder per cow, each holding a few photos:

```
looking_for/
    Bella/   bella1.jpg  bella2.jpg  bella3.jpg
    Greta/   greta1.jpg  greta2.jpg
```

The folder name is the cow's name; more photos = a more reliable fingerprint.
**Tip:** photograph from above and at an angle (back and upper sides) — that is what the drone sees.

## ⚙️ Settings

In [ ]:
from pathlib import Path

# Work from the repository root, whether Jupyter was started here or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DETECTOR_WEIGHTS = ROOT / "weights" / "best.pt"        # your trained detector
GALLERY_DIR = ROOT / "looking_for"                     # one sub-folder per known cow
DRONE_IMAGE = ROOT / "photos" / "my_drone_photo.jpg"   # <-- EDIT: the drone photo to analyse

print("Detector exists?", DETECTOR_WEIGHTS.exists())
print("Gallery exists? ", GALLERY_DIR.exists())

## 📦 Step 1 — Load the detector and the fingerprint model

In [ ]:
from cow_identifier import reid
from cow_identifier.common import load_detector

detector = load_detector(DETECTOR_WEIGHTS)   # loaded, not retrained
embedder = reid.load_embedder()              # ResNet50; downloaded once, then cached

## 🗂️ Step 2 — Build the gallery of known cows

In [ ]:
gallery = reid.build_gallery(embedder, GALLERY_DIR)
print(f"\nGallery ready with {len(gallery)} cow(s):", list(gallery))

## 🎯 Step 3 — Identify the cows in a drone photo

Each detected cow gets its best-matching name plus the full ranking. Read the **gap** between 1st and
2nd place, not the raw score. `Bella:0.81 Greta:0.42` is confident; `Bella:0.71 Greta:0.70` is a guess.

Both thresholds start at `0` (always give a name). Once you have seen real scores, raise
`MIN_MARGIN` so near-ties are flagged as *uncertain* (orange boxes).

In [ ]:
import cv2
import matplotlib.pyplot as plt

MATCH_THRESHOLD = 0.0   # minimum similarity to accept a name
MIN_MARGIN = 0.0        # minimum lead of 1st over 2nd place to accept a name

found = reid.identify_cows_in_image(
    detector, embedder, gallery, DRONE_IMAGE,
    conf=0.4, match_threshold=MATCH_THRESHOLD, min_margin=MIN_MARGIN,
)

annotated = reid.draw_identifications(cv2.imread(str(DRONE_IMAGE)), found)
plt.figure(figsize=(15, 11))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"{len(found)} cow(s) — best matches")
plt.show()

print("Per-cow ranking (this is what tells you whether it really works):\n")
for i, cow in enumerate(found, 1):
    ranking = "   ".join(f"{name}:{score:.2f}" for name, score in cow["ranked"])
    print(f"  Cow #{i}: {ranking}   (margin {cow['margin']:.2f})")

## 📝 Reading the results

If the scores are bunched together, that is information, not failure: the ground-level reference photos
do not match the drone's top-down view well enough yet. The fix is reference photos that include the
back and top of each cow — or, later, a fingerprint model trained specifically on cows.